In [10]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import scipy as sp

In [11]:
#Define constants
amount_of_nodes = 5

In [ ]:
class NARMA:
    def __init__(self, degree_, alpha_, beta_, gamma_, delta_):
        self.alpha = alpha_
        self.beta = beta_
        self.gamma = gamma_
        self.delta = delta_
        self.NARMA_degree = degree_
        self.NARMA_outputs = [0] * degree_
        self.NARMA_inputs = [0] * degree_

        self.name = f'NARMA({degree_}, {alpha_}, {beta_}, {gamma_}, {delta_})'

    def update_NARMA_constant(self, alpha_=None, beta_=None, gamma_=None, delta_=None):
        if alpha_:
            self.alpha = alpha_
        if beta_:
            self.beta = alpha_
        if gamma_:
            self.gamma = gamma_
        if delta_:
            self.delta = delta_

    def update_NARMA_degree(self, degree_):
        if degree_ % 1 == 0:
            self.NARMA_degree = degree_
            self.NARMA_outputs = [0] * degree_
            self.NARMA_inputs = [0] * degree_
        else:
            print("Degree has to be a whole number")

    def reset_NARMA(self):
        self.NARMA_outputs = [0] * self.NARMA_degree
        self.NARMA_inputs = [0] * self.NARMA_degree

    def perform_NARMA(self, inputs_):
        NARMA_vals_ = []

        for input in inputs_:
            NARMA_new_value_ = self.alpha * self.NARMA_outputs[0] + self.beta * self.NARMA_outputs[0] * sum(self.NARMA_outputs) + self.gamma * self.NARMA_inputs[-1] * self.NARMA_inputs[0] + self.delta

            NARMA_vals_.append(NARMA_new_value_)
            self.NARMA_outputs = [NARMA_new_value_] + self.NARMA_outputs[:-1]
            self.NARMA_inputs = [input] + self.NARMA_inputs[:-1]

        return NARMA_vals_

class Reservoir:
    def __init__(self, reservoir_size_=4, feedback_gain_=1.8, input_gain_=0.01, spatial_multiplexing_=None):
        self.reservoir_size = reservoir_size_
        self.feedback_gain = feedback_gain_
        self.input_gain = input_gain_
        if spatial_multiplexing_:
            self.spatial_multiplexing = spatial_multiplexing_
        else:
            self.spatial_multiplexing = False

        #Set initial state
        if spatial_multiplexing_:
            self.reservoir_state = np.random.uniform(-1,1,(spatial_multiplexing_,reservoir_size_))
        else:
            self.reservoir_size = np.random.uniform(-1,1,(reservoir_size_,))
        
        self.task_counter = 1
        self.results = {}

    def add_performance_task(self, performance_task_):
        taskname_ = f"task{self.task_counter}"
        self.task_counter += 1
        self.results[taskname_] = performance_task_
            
    #Resets the reservoir state to a random initial state
    def reset_reservoir_state(self):
        self.reservoir_state = np.random.uniform(-1,1,(self.reservoir_size,))

    #Updates the echo state network constants
    def update_reservoir_constant(self, reservoir_size_=None, feedback_gain_=None, input_gain_=None, spatial_multiplexing_=None):
        if reservoir_size_:
            self.reservoir_size = reservoir_size_
        if feedback_gain_:
            self.feedback_gain = feedback_gain_
        if input_gain_:
            self.input_gain = input_gain_
        if spatial_multiplexing_:
            self.spatial_multiplexing = spatial_multiplexing_

    #Generates various constant matrices and vectors used in the updation of reservoir
    def generate_scaling(self):
        #Matrix W is scaled to have spectral radius of 1
        unscaled_W_ = np.random.uniform(-1,1,(self.reservoir_size,self.reservoir_size))
        scaling_factor_W_ = 1 / np.abs(np.linalg.eigvals(unscaled_W_)).max()
        self.matrix_W = scaling_factor_W_*unscaled_W_

        self.vector_b = np.random.uniform(-1,1,(self.reservoir_size,))
        self.vector_v = np.random.uniform(-1,1,(self.reservoir_size,))

    #Generates inputs for reservoir preparation, training and testing
    def generate_inputs(self, prep_train_test_split_):
        self.prep_inputs = np.random.uniform(-1,1,(prep_train_test_split_[0],))
        self.train_inputs = np.random.uniform(-1,1,(prep_train_test_split_[1],))
        self.test_inputs = np.random.uniform(-1,1,(prep_train_test_split_[2],))

    #Calculates the first and second moments
    def calculate_moments(data_):
        if len(np.shape(data_)) == 2:
            [[np.mean(x), np.mean(x**2)] for x in data_]
        else:
            return [np.mean(data_), np.mean(data_**2)]

    def measure_state(self):
        return self.reservoir_state

    #Updates reservoir state
    def update_reservoir(self, input_):
        for i, network in enumerate(self.reservoir_state):
            self.reservoir_state[i] = np.tanh(self.feedback_gain * self.matrix_W @ network + self.vector_b + self.input_gain * input_ * self.vector_v)

    def prep_reservoir(self):
        for prep_input in self.prep_inputs:
            Reservoir.update_reservoir(prep_input)

    def train_reservoir(self):
        measured_observables_ = []
        for train_input in self.train_inputs:
            Reservoir.update_reservoir(train_input)
            
            measured_observables_.append(np.append(self.reservoir_state, 1))

        measured_observables_ = np.asarray(measured_observables_)

        #!!!!
        self.trained_weights = np.linalg.inv(measured_observables_.T @ measured_observables_) @ measured_observables_.T @ np.append(self)

    #Updates reservoir state and measures it's observables
    def update_and_measure_reservoir(self, phase_):
        if phase_ == "prep":
            for train_input in self.train_inputs:
                Reservoir.update_reservoir(train_input)

        elif phase_ == "train":
            measured_observables_ = []
            for train_input in self.train_inputs:
                Reservoir.update_reservoir(train_input)
                measured_observables_.append(Reservoir.calculate_moments(train_input) + [1])
            return np.asarray(measured_observables_)
        
        elif phase_ == "test":
            measured_observables_ = []
            for test_input in self.test_inputs:
                Reservoir.update_reservoir(test_input)             
                measured_observables_.append(Reservoir.calculate_moments(test_input) + [1])
            return np.asarray(measured_observables_)

    def NMSE(true_outputs_, generated_outputs_):
        return sum([(x-y)**2 for x,y in zip(true_outputs_, generated_outputs_)])/sum([x**2 for x in true_outputs_])
    
    def evolve_reservoir(self, task_, phase_):

        performance_task_ = self.results[task_]

        #Full simulation
        if phase_ == "full":
            #Preparation
            performance_task_.perform_NARMA(self.prep_inputs)
            Reservoir.update_and_measure_reservoir("prep")

            #Training
            train_observables_ = Reservoir.update_and_measure_reservoir("train")
            train_true_outputs_ = performance_task_.perform_NARMA(self.train_inputs)
            self.trained_weights = np.linalg.inv(train_observables_.T @ train_observables_) @ train_observables_.T @ train_true_outputs_

            #Testing
            test_observables_ = Reservoir.update_and_measure_reservoir("test")
            test_outputs_ = test_observables_ @ self.trained_weights
            test_true_outputs_ = performance_task_.perform_NARMA(self.test_inputs)

            self.results[task_] = (performance_task_.name, NMSE(test_true_outputs_, test_outputs_))

        #Preparation phase, where the dependence on the initial state of reservoir is removed 
        elif phase_ == "prep":
            performance_task_.perform_NARMA(self.prep_inputs)
            Reservoir.update_and_measure_reservoir("prep") 
        #Training phase, where the weights are trained
        elif phase_ == "train":
            train_observables_ = Reservoir.update_and_measure_reservoir("train")
            train_true_outputs_ = performance_task_.perform_NARMA(self.train_inputs)
            self.trained_weights = np.linalg.inv(train_observables_.T @ train_observables_) @ train_observables_.T @ train_true_outputs_
        #Testing phase
        elif phase_ == "test":          
            test_observables_ = Reservoir.update_and_measure_reservoir("test")
            test_outputs_ = test_observables_ @ self.trained_weights
            test_true_outputs_ = performance_task_.perform_NARMA(self.test_inputs)

            self.results[task_] = (performance_task_.name, NMSE(test_true_outputs_, test_outputs_))
        else:
            print("No such phase exists!")    










        


In [15]:
jeps = NARMA(2, 0,1, 0.1, 0.1)
jeps